# Transform CBS Data

1. Read bronze `CBS` table
1. Drop unwanted columns
1. Standardise column names using snake_case 
1. Filter out rows where `txn_id` is null (business key validation)
1. Remove duplicate records
1. Write the transformed data to silver `circuits` table


In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
import pyspark.sql.functions as F

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.cbs"
silver_table = f"{catalog_name}.{silver_schema}.cbs"

In [0]:
cbs_df = spark.read.table(bronze_table).filter(F.col("batch_id")==v_batch_id)

In [0]:
cbs_renamed = cbs_df.withColumnsRenamed({"TransactionID":"txn_id","BillerID":"biller_id","CustomerID":"customer_id","CreditAmount":"credit_amount","CreditStatus":"credit_status","BankReferenceNo":"bank_ref_no","CreditTimestamp":"credit_time"})


In [0]:
cbs_final = cbs_renamed.filter(F.col("txn_id").isNotNull()).dropDuplicates()

In [0]:
cbs_final = cbs_final.fillna("NA", subset=["bank_ref_no"])

In [0]:
cbs_final = cbs_final.withColumn("created_timestamp", F.current_timestamp()) \
                    .withColumn("updated_timestamp", F.current_timestamp())

In [0]:
silver_table = "payment_app.silver.cbs"

if not spark.catalog.tableExists(silver_table):

    (
        cbs_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

else:

    cbs_final.createOrReplaceTempView("vw_cbs_final")

    spark.sql(f"""
        MERGE INTO {silver_table} AS tgt
        USING vw_cbs_final AS src
        ON tgt.txn_id = src.txn_id

        WHEN MATCHED
        AND src.batch_id >= tgt.batch_id
        THEN UPDATE SET
            tgt.biller_id          = src.biller_id,
            tgt.customer_id        = src.customer_id,
            tgt.credit_amount      = src.credit_amount,
            tgt.credit_status      = src.credit_status,
            tgt.bank_ref_no        = src.bank_ref_no,
            tgt.credit_time        = src.credit_time,
            tgt.ingestion_timestamp= src.ingestion_timestamp,
            tgt.source_file        = src.source_file,
            tgt.batch_id           = src.batch_id,
            tgt.updated_timestamp  = current_timestamp()

        WHEN NOT MATCHED
        THEN INSERT (
            txn_id,
            biller_id,
            customer_id,
            credit_amount,
            credit_status,
            bank_ref_no,
            credit_time,
            ingestion_timestamp,
            source_file,
            batch_id,
            created_timestamp,
            updated_timestamp
        )
        VALUES (
            src.txn_id,
            src.biller_id,
            src.customer_id,
            src.credit_amount,
            src.credit_status,
            src.bank_ref_no,
            src.credit_time,
            src.ingestion_timestamp,
            src.source_file,
            src.batch_id,
            src.created_timestamp,
            src.updated_timestamp
        )
    """)

In [0]:
display(spark.table(silver_table))